In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install folium --quiet

import folium
import pandas as pd
import numpy as np
from folium.plugins import MarkerCluster
from folium.plugins import MousePosition
from folium.features import DivIcon

print("✅ Libraries loaded!")
print("Folium version:", folium.__version__)

✅ Libraries loaded!
Folium version: 0.20.0


In [3]:
# Load spacex launch geo dataset
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv"
spacex_df = pd.read_csv(url)
print("Shape:", spacex_df.shape)
print("\nColumns:", spacex_df.columns.tolist())
spacex_df.head()

Shape: (56, 13)

Columns: ['Flight Number', 'Date', 'Time (UTC)', 'Booster Version', 'Launch Site', 'Payload', 'Payload Mass (kg)', 'Orbit', 'Customer', 'Landing Outcome', 'class', 'Lat', 'Long']


,Flight Number,Date,Time (UTC),Booster Version,Launch Site,Payload,Payload Mass (kg),Orbit,Customer,Landing Outcome,class,Lat,Long
0,1,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Failure (parachute),0,28.562302,-80.577356
1,2,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel o...",0.0,LEO (ISS),NASA (COTS) NRO,Failure (parachute),0,28.562302,-80.577356
2,3,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2+,525.0,LEO (ISS),NASA (COTS),No attempt,0,28.562302,-80.577356
3,4,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356
4,5,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677.0,LEO (ISS),NASA (CRS),No attempt,0,28.562302,-80.577356


In [4]:
# Get unique launch sites with coordinates
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
print(launch_sites_df)

    Launch Site        Lat        Long
0   CCAFS LC-40  28.562302  -80.577356
1  CCAFS SLC-40  28.563197  -80.576820
2    KSC LC-39A  28.573255  -80.646895
3   VAFB SLC-4E  34.632834 -120.610745


In [5]:
# Create base map centered on NASA Kennedy Space Center
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)

# Add circle markers for each launch site
for index, row in launch_sites_df.iterrows():
    folium.Circle(
        location=[row['Lat'], row['Long']],
        radius=1000,
        color='#d35400',
        fill=True
    ).add_to(site_map)

    folium.Marker(
        location=[row['Lat'], row['Long']],
        icon=DivIcon(
            icon_size=(20,20),
            icon_anchor=(0,0),
            html='<div style="font-size: 12px; color:#d35400;"><b>%s</b></div>' % row['Launch Site']
        )
    ).add_to(site_map)

# Save map
site_map.save('launch_sites_map.html')
print("✅ Task 1 Done!")
site_map

✅ Task 1 Done!


In [6]:
# Select relevant columns
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]

# Create marker cluster map
site_map2 = folium.Map(location=nasa_coordinate, zoom_start=5)
marker_cluster = MarkerCluster()

# Color based on success/failure
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'

for index, record in spacex_df.iterrows():
    marker = folium.Marker(
        location=[record['Lat'], record['Long']],
        icon=folium.Icon(color=assign_marker_color(record['class']),
                        icon='info-sign')
    )
    marker_cluster.add_child(marker)

site_map2.add_child(marker_cluster)
site_map2.save('launch_outcomes_map.html')
print("✅ Task 2 Done!")
site_map2

✅ Task 2 Done!


In [7]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    R = 6373.0
    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c

# CCAFS SLC 40 coordinates
launch_site_lat = 28.562302
launch_site_lon = -80.577356

# Nearest coastline point
coastline_lat = 28.56367
coastline_lon = -80.56799

distance_coastline = calculate_distance(launch_site_lat, launch_site_lon,
                                         coastline_lat, coastline_lon)
print(f"Distance to coastline: {distance_coastline:.2f} km")

Distance to coastline: 0.93 km


In [8]:
site_map3 = folium.Map(location=[launch_site_lat, launch_site_lon], zoom_start=12)

# Launch site marker
folium.Marker(
    location=[launch_site_lat, launch_site_lon],
    icon=folium.Icon(color='blue', icon='info-sign'),
    popup='CCAFS SLC 40'
).add_to(site_map3)

# Coastline marker
folium.Marker(
    location=[coastline_lat, coastline_lon],
    icon=folium.Icon(color='red', icon='info-sign'),
    popup='Coastline'
).add_to(site_map3)

# Draw line between them
folium.PolyLine(
    locations=[[launch_site_lat, launch_site_lon],
               [coastline_lat, coastline_lon]],
    weight=2,
    color='blue'
).add_to(site_map3)

# Add distance label
folium.Marker(
    location=[(launch_site_lat + coastline_lat)/2,
              (launch_site_lon + coastline_lon)/2],
    icon=DivIcon(
        icon_size=(200,36),
        icon_anchor=(0,0),
        html=f'<div style="font-size:12px; color:blue;"><b>{distance_coastline:.2f} km</b></div>'
    )
).add_to(site_map3)

site_map3.save('distance_map.html')
print("✅ Task 4 Done!")
site_map3

✅ Task 4 Done!


In [9]:
import shutil
shutil.copy('launch_sites_map.html', '/content/drive/MyDrive/launch_sites_map.html')
shutil.copy('launch_outcomes_map.html', '/content/drive/MyDrive/launch_outcomes_map.html')
shutil.copy('distance_map.html', '/content/drive/MyDrive/distance_map.html')
print("✅ All maps saved to Google Drive!")

✅ All maps saved to Google Drive!
